# Funciones y decoradores en Python
Este cuaderno contiene ejemplos prácticos y explicaciones (en español) sobre:

- funciones simples
- funciones de orden superior
- decoradores simples y parametrizados
- decoradores para métodos de clase/instancia
- @staticmethod, @classmethod, @property
- uso de functools.wraps
- ejemplos de memoización, temporizador y reintento

Ejecuta cada celda para ver los ejemplos en acción.

## Funciones simples y de orden superior

In [ ]:
def suma(a, b):
    """Devuelve la suma de a y b."""
    return a + b


def aplicar_funcion(f, lista):
    """Aplica f a cada elemento de lista y devuelve la lista resultante."""
    return [f(x) for x in lista]

print('suma(2,3)=', suma(2,3))
print('aplicar_funcion(lambda x: x*2, [1,2,3]) =', aplicar_funcion(lambda x: x*2, [1,2,3]))

## Decorador simple (log)

In [ ]:
from functools import wraps

def log(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f'[LOG] Llamando {func.__name__} con args={args}, kwargs={kwargs}')
        resultado = func(*args, **kwargs)
        print(f'[LOG] {func.__name__} devolvió {resultado}')
        return resultado
    return wrapper

@log
def resta(a, b):
    return a - b

resta(10, 4)

## Decorador parametrizado (reintentos)

In [ ]:
import time
from functools import wraps

def retry(times=3, delay=0.5):
    """Reintenta la función `times` veces si lanza una excepción."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for i in range(1, times+1):
                try:
                    print(f'Intento {i} de {times} para {func.__name__}')
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    print(f'  Falló: {e}; esperando {delay} s antes del siguiente intento')
                    time.sleep(delay)
            print('Agotados los reintentos.')
            raise last_exc
        return wrapper
    return decorator

import random

@retry(times=4, delay=0.2)
def azarosa():
    if random.random() < 0.7:
        raise ValueError('fallo aleatorio')
    return 'éxito'

try:
    print('Resultado azarosa():', azarosa())
except Exception as e:
    print('La función finalmente falló:', e)

## Decoradores y decoradores de métodos: @staticmethod, @classmethod, @property

In [ ]:
class Cuenta:
    def __init__(self, titular, saldo=0):
        self.titular = titular
        self._saldo = saldo

    def depositar(self, cantidad):
        self._saldo += cantidad
        return self._saldo

    def retirar(self, cantidad):
        if cantidad > self._saldo:
            raise ValueError('Fondos insuficientes')
        self._saldo -= cantidad
        return self._saldo

    @property
    def saldo(self):
        """Permite acceder a saldo como atributo de solo lectura."""
        return self._saldo

    @staticmethod
    def moneda():
        return 'COP'  # ejemplo

    @classmethod
    def desde_dict(cls, data):
        return cls(data['titular'], data.get('saldo', 0))

c = Cuenta('María', 100)
print('saldo:', c.saldo)
print('moneda:', Cuenta.moneda())
c2 = Cuenta.desde_dict({'titular':'Luis', 'saldo':50})
print('c2 titular y saldo:', c2.titular, c2.saldo)

## Temporizador y memoización

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        t1 = time.perf_counter()
        print(f'[TIMER] {func.__name__} tardó {t1-t0:.6f} s')
        return result
    return wrapper


def memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args):
        if args in cache:
            return cache[args]
        res = func(*args)
        cache[args] = res
        return res
    return wrapper

@timer
@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print('fib(20)=', fib(20))

## Decorador para métodos de instancia (ejemplo: requires_login)

In [ ]:
from functools import wraps

def requires_login(method):
    @wraps(method)
    def wrapper(self, *args, **kwargs):
        if not getattr(self, 'logged_in', False):
            raise PermissionError('Usuario no autenticado')
        return method(self, *args, **kwargs)
    return wrapper

class Servicio:
    def __init__(self):
        self.logged_in = False

    def login(self):
        self.logged_in = True

    @requires_login
    def operar(self, x):
        return f'Operando con {x}'

s = Servicio()
try:
    s.operar(5)
except Exception as e:
    print('Error esperado (no autenticado):', e)
s.login()
print('Después de login:', s.operar(5))

---

### Notas finales

- Usa `@wraps` para preservar metadata de la función original.
- Los decoradores parametrizados retornan un decorador.
- Puedes combinar decoradores: `@timer\n@memoize`.

Si quieres que incluya más ejemplos (por ejemplo: decoradores con clases, decoradores para métodos estáticos, o ejemplos de typing y mypy), dime y lo añado.